# 01 Acquire

One section per modality. Each section requests the data, saves the raw response unchanged to its own folder in `data/raw/`, then checks what was saved.
The old Adzuna code is in `archive/01_acquire_adzuna_archive.ipynb`.

# All imports

In [ ]:
import requests
import os
import json
import time
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv

# Load the keys from the .env file

In [ ]:
load_dotenv("../../.env")    # the notebook runs in notebooks/, so go up to the top-level folder to get .env file
print("keys loaded")

---
# 1. API: TheirStack Jobs API
Saves raw responses to `data/raw/theirstack/`.

Building the request

In [ ]:
url = "https://api.theirstack.com/v1/jobs/search"
headers = {"Authorization": f"Bearer {os.environ['THEIRSTACK_API_KEY']}"}
roles = ["data analyst", "data engineer", "data scientist"]
os.makedirs("../data/raw/theirstack", exist_ok=True)

Loops each role in Canada and saves one file per role. Skips files already saved (here or from the Oct 16 pull in `data/raw/`) to save API credits

In [ ]:
page = 1

for role in roles:
    body = {
        "posted_at_max_age_days": 90,
        "job_country_code_or": ["CA"],
        "job_title_or": [role],
        "limit": 25,
        "page": page,
    }
    name = f"theirstack_ca_{role.replace(' ', '_')}_p{page}.json"
    filename = f"../data/raw/theirstack/{name}"

    if os.path.exists(filename) or os.path.exists(f"../data/raw/{name}"):
        print("already saved, skipping:", name)
        continue

    response = requests.post(url, headers=headers, json=body)
    print(datetime.now(), "|", role, "| status", response.status_code)
    if response.status_code != 200:
        print(response.text)
        break

    data = response.json()
    with open(filename, "w") as f:
        json.dump(data, f, indent=2)
    print("saved", len(data["data"]), "jobs to", filename)
    time.sleep(1)

Check: one row per posting

In [ ]:
files = sorted(f"../data/raw/theirstack/{n}" for n in os.listdir("../data/raw/theirstack") if n.endswith(".json"))
if not files:   # fall back to the Oct 16 pulls
    files = sorted(f"../data/raw/{n}" for n in os.listdir("../data/raw") if n.startswith("theirstack_ca_") and n.endswith("_p1.json"))

rows = []
for name in files:
    with open(name) as f:
        for job in json.load(f)["data"]:
            job["source_file"] = os.path.basename(name)
            rows.append(job)

theirstack = pd.DataFrame(rows)
print(len(files), "files |", len(theirstack), "rows |", theirstack["id"].nunique(), "unique ids")
theirstack[["id", "job_title", "company", "location", "date_posted", "source_file"]].head()

---
# 2. Web scraping: Job Bank wages by occupation

---
# 3. Database / SQL: Job Bank open data

---
# 4. Technique beyond class (to be decided)